In [215]:
import reframed
import pandas as pd
from pathlib import Path
import numpy as np
import sys

sys.path.append('../../code/7_GEM_reconstruction')

# from ng_utils import *
# from ng_tests import *
# from model_qc_and_polish import curate_model, remove_duplicated_metabolites, remove_duplicated_reactions, test_balance


from collections import defaultdict


In [216]:
repo_folder = Path("../..")
data_folder = repo_folder / "data" / "7_GEM_reconstruction"
gapfilled_folder = data_folder / 'models/gapfilled_FBA'
polished_folder = data_folder / 'models/polished'
# genome_folder = data_folder / 'genomes'

In [217]:
model_abbrvs = ['At', 'Ct', 'Ml', 'Oa']

In [218]:
model_dict = {}
for abbrv in model_abbrvs:
    model_path = gapfilled_folder / f'{abbrv}_gapfilled.xml'
    model = reframed.load_cbmodel(str(model_path))
    model_dict[abbrv] = model

## Fix missing charges

In [219]:
charge_fn = data_folder / 'missing_metabolite_charges_bigg.csv'
charge_dict = pd.read_csv(charge_fn, index_col=0)['charge'].to_dict()

In [220]:
fixed_charges = {
    'ACP': -1,
    'fad': -2,
    'fadh2': -2,
    'ddcaACP': -1,
    'hd2coa': -4,
    'ddecoa': -4,
    'focytc': 0,
    'ode2coa': -4,
    # 'hdd7coa': 2,
    # '3oodecoa': 2,
    # '3hodecoa':2,
    'dde2coa': -4,
    'ddecoa': -5,
    # '3htdccoa': -4,
    '3hdd5coa': -4,
    'hdd4_2_coa': -4,
    "lnlc2coa": -4,
    'dec4coa':-4,
    'ptal': 0,
    'dann': 2,
    'dtbt':0,
    'btn': 0,
    'fdxo_2_2': 6,
    'fdxrd': 4,
    'fdxox': 5,
    'malACP': -2,
    'dc2coa': -4,
    'nonacoa': -2,
    'nona': 1,
    'poctacoa': -4,
    'php2coa': -4,
    'R_3hphpa': 1,
    'phxa2coa': -4,
    'ppt2coa': -6,
    '4atb2coa': -2,
    'decoa': -3,
    'R_3hnonaa': -1,
    'pta': -1,
    'pdcacoa': -4,
    'pocta2coa': -4,
    'R_3hpocta':-1,
    'tdecoa': -4,
    'R_3hdd6e': -1,
    'R_3hppta':-3,
    'R_3hphpa':-1,
    'R_3hphxa': -1,
    'ssaltpp': -3,
    'hp2coa': -4,
    'R_3hcmrs7e': -2,
    'R_3hhpa':-1,
    'lnlccoa':-2,
    'lnlc':1,
    'fe3dcit':-3,
    'seramp':0,
    'tag6p__D':-2,
    'mi3p__D':-2,
    'm2bcoa':0,
    '3h4atb': 1,
    'fmn':-2,
    'fmnh2':-2,
    'ribflv': 0,
    'hmbpp': -4,
    '23dhbzs2':-1,
    '23dhbzs3': -1,
    'fe3dhbzs3': 3,
    'trnaglu': 0,
    'cinmcoa': -4,
    # 'cinnm':-4
    # 'dmlz': -1,
    # 'db4p': -2,
    # '4r5au':-2,
    'salchs4': -1,
    'feenter': 2,
    'salc': -1,
    'dmso2': 4,
    'phenona': -3,
    'phehpa': -1,
    'abg4': -3,
    'val__D': -2,
    '3hpnonacoa': -4,
    '6ath2coa': -4,
    'hptcoa': -4,
}

equal = {
    'M_dde2coa_c': ['M_R_3hcddec5ecoa_c', 'M_3hddccoa_c', 'M_3oddccoa_c', 'M_decoa_c'],
    'M_3hpnonacoa_c': ['M_pnona2coa_c', 'M_R_3hpnonacoa_c', 'M_3opnonacoa_c', 'M_phpcoa_c'],
    'M_ddecoa_c': ['M_3otdccoa_c', 'M_3htdccoa_c', 'M_tde2coa_c', 'M_R_3hcmrs7ecoa_c'], 
    'M_3hdd5coa_c': ['M_tded5_2_coa_c', 'M_tded5coa_c', 'M_3ohdd7coa_c', 'M_3hhdd7coa_c', 'M_hdd7_2_coa_c'],
    # 'M_td2coa_c': ['M_tde_2Z_coa_c'],
    "M_lnlc2coa_c": ["M_3hlnlccoa_c", 'M_3olnlccoa_c', 'M_hd_7_10_coa_c', 'M_hd710_2_coa_c', 'M_3hhd710coa_c',
                     'M_3ohd710coa_c', 'M_td_5_8_coa_c', 'M_td58_2_coa_c', 'M_R_3htd58coa_c'],
    'M_dc2coa_c': ['M_R3hdec4coa_c', 'M_dec4_2_coa_c', 'M_3hdcoa_c'],
    'M_dec4coa_c': ['M_3odd6coa_c', 'M_3hdd6coa_c', 'M_dd6_2_coa_c', 'M_dd_3_6_coa_c', 'M_R_3hdd6coa_c',
                    'M_dd_3_6_coa_c', 'M_3ohd58coa_c', 'M_3hhd58coa_c'],
    'M_pocta2coa_c': ['M_R_3hpoctacoa_c', 'M_3hpoctacoa_c', 'M_3opoctacoa_c', 'M_phxacoa_c'],
    'M_php2coa_c': ['M_3hphpcoa_c', 'M_R_3hphpcoa_c','M_3ophpcoa_c', 'M_pptcoa_c'],
    # 'M_pro__D_c':['M_1p2cbxl_c'],
    'M_6ath2coa_c': ['M_R_3h6athcoa_c', 'M_3h6athcoa_c'],
    'M_hptcoa_c' : ['M_3ononacoa_c', 'M_3hnnacoa_c', 'M_nona2coa_c', 'M_R_3hnonacoa_c'],
    'M_hp2coa_c': ['M_3hhpcoa_c', 'M_R_3hhpcoa_c', 'M_3ohpcoa_c', 'M_ptcoa_c', 
                    'M_pt2coa_c', 'M_3hptcoa_c', 'M_3optcoa_c', 'M_ppcoa_c'],
    'M_4atb2coa_c': ['M_3h4atbcoa_c'],
    'M_hd2coa_c': ['M_3hhdccoa_c', 'M_3ohdccoa_c'],
    'M_ode2coa_c': ['M_3hodecoa_c', 'M_3oodecoa_c', 'M_hdd7coa_c'],
    'M_hdd4_2_coa_c': ['M_3hhdd4coa_c', 'M_3ohdd4coa_c', 'M_tde_2Z_coa_c'],
    
    'M_poctacoa_c': ['M_3opdecacoa_c', 'M_3hpdecacoa_c', 'M_pdca2coa_c', 'M_td2coa_c'],
    # 'M_decoa_c': ['M_3oddccoa_c'],
    'M_phxa2coa_c': ['M_R_3hphxacoa_c', 'M_3hphxacoa_c', 'M_3ophxacoa_c', 'M_pbcoa_c'],
    'M_ppt2coa_c': ['M_R_3hpptcoa_c', 'M_3hpptcoa_c', 'M_3opptcoa_c', 'M_phppcoa_c'],
}

for key, key_lst in equal.items():
    charge = fixed_charges.get(key[2:-2], None)
    if charge is not None:
        for short_id in key_lst:
            fixed_charges[short_id[2:-2]] = charge
            # charge_dict[short_id[2:-2]] = charge
            print(f'Setting fixed charge for {short_id[2:-2]} to {charge}')
    else:
        print(f'No fixed charge for {key} to set equal to {key_lst}')
charge_dict.update(fixed_charges)

Setting fixed charge for R_3hcddec5ecoa to -4
Setting fixed charge for 3hddccoa to -4
Setting fixed charge for 3oddccoa to -4
Setting fixed charge for decoa to -4
Setting fixed charge for pnona2coa to -4
Setting fixed charge for R_3hpnonacoa to -4
Setting fixed charge for 3opnonacoa to -4
Setting fixed charge for phpcoa to -4
Setting fixed charge for 3otdccoa to -5
Setting fixed charge for 3htdccoa to -5
Setting fixed charge for tde2coa to -5
Setting fixed charge for R_3hcmrs7ecoa to -5
Setting fixed charge for tded5_2_coa to -4
Setting fixed charge for tded5coa to -4
Setting fixed charge for 3ohdd7coa to -4
Setting fixed charge for 3hhdd7coa to -4
Setting fixed charge for hdd7_2_coa to -4
Setting fixed charge for 3hlnlccoa to -4
Setting fixed charge for 3olnlccoa to -4
Setting fixed charge for hd_7_10_coa to -4
Setting fixed charge for hd710_2_coa to -4
Setting fixed charge for 3hhd710coa to -4
Setting fixed charge for 3ohd710coa to -4
Setting fixed charge for td_5_8_coa to -4
Setting

In [221]:
STILL_MISSING = []

In [222]:
def fix_charge(model):
    for short_id, charge in fixed_charges.items():
        for end in ['_c', '_p', '_e']:
            met_id = f'M_{short_id}{end}'
            try:
                met = model.metabolites[met_id]
                # print(f'Setting fixed CHARGE for {met.id} in model {model.id} to {charge}')
            except KeyError:
                continue
            else:
                met.metadata['CHARGE'] = str(int(float(charge)))

    # Fill in missing charges from charge_dict
    for m_id in model.metabolites:
        met = model.metabolites[m_id]
        try:
            charge = int(float(met.metadata.get('CHARGE', np.nan)))
        except ValueError:
            charge = np.nan
        if np.isnan(charge):
            short_id = met.id[2:-2]
            new_charge = charge_dict.get(short_id, np.nan)
            if np.isfinite(new_charge):
                # print(f'Setting CHARGE for {short_id} in model {model.id} to {new_charge}')
                met.metadata['CHARGE'] = str(int(float(new_charge)))
            else:
                print(f'No CHARGE for {short_id} in model {model.id} or in charge dict')
                print(charge, new_charge)
                # print(met.metadata)
                STILL_MISSING.append(short_id)

    for m_id in model.metabolites:
        met = model.metabolites[m_id]
        try:
            met.metadata['CHARGE'] = str(int(float(met.metadata['CHARGE'])))
        except KeyError:
            print(f'No CHARGE for {met.id} in model {model.id} after fixing charges.')

# Small curations

In [223]:
def small_curations(model_dict):
    # At
    at = model_dict['At']
    at.remove_reaction('R_BTS2')
    at.reactions['R_EX_glc__D_e'].lb = 0
    # Ct
    ct = model_dict['Ct']
    ct.reactions['R_EX_ac_e'].lb = 0
    # Ml
    ml = model_dict['Ml']
    ml.reactions['R_EX_cys__L_e'].lb = 0
    ml.reactions['R_EX_btn_e'].lb = 0
    ml.reactions['R_EX_thm_e'].lb = 0
    # Oa
    oa = model_dict['Oa']
    oa.reactions['R_EX_thm_e'].lb = 0
    oa.reactions['R_EX_glc__D_e'].lb = 0



# Add gene annotations

In [224]:
model_dict['Oa'].genes

AttrOrderedDict([('G_WKT91696_1', G_WKT91696_1),
                 ('G_spontaneous', G_spontaneous),
                 ('G_WKT92582_1', G_WKT92582_1),
                 ('G_WKT92581_1', G_WKT92581_1),
                 ('G_WKT94662_1', G_WKT94662_1),
                 ('G_WKT91561_1', G_WKT91561_1),
                 ('G_WKT91855_1', G_WKT91855_1),
                 ('G_WKT95715_1', G_WKT95715_1),
                 ('G_WKT91653_1', G_WKT91653_1),
                 ('G_WKT94599_1', G_WKT94599_1),
                 ('G_WKT94047_1', G_WKT94047_1),
                 ('G_WKT95337_1', G_WKT95337_1),
                 ('G_WKT95210_1', G_WKT95210_1),
                 ('G_WKT94098_1', G_WKT94098_1),
                 ('G_WKT93812_1', G_WKT93812_1),
                 ('G_WKT91538_1', G_WKT91538_1),
                 ('G_WKT91537_1', G_WKT91537_1),
                 ('G_WKT91536_1', G_WKT91536_1),
                 ('G_WKT91822_1', G_WKT91822_1),
                 ('G_WKT94325_1', G_WKT94325_1),
                 (

In [225]:
def add_ncbiprotein_annotations(model_dict):
    # Add NCBI protein IDs to gene annotations
    for abbrv, model in model_dict.items():
        i = 0
        for gene in model.genes.values():
            protein_id = gene.id[2:].replace('_', '.')
            if protein_id[:3] in ['WKL', 'WKT']:
                gene.metadata['ncbiprotein'] = protein_id
                i += 1
        print(f'Added {i} gene annotations to model {abbrv}')

    

# Save models

In [226]:
small_curations(model_dict)
add_ncbiprotein_annotations(model_dict)
for abbrv, model in model_dict.items():
    
    fix_charge(model)
    polished_path = polished_folder / f'{abbrv}.xml'
    reframed.save_cbmodel(model, str(polished_path))

Added 1411 gene annotations to model At
Added 1353 gene annotations to model Ct
Added 1021 gene annotations to model Ml
Added 1387 gene annotations to model Oa


# Code for testing purposes

In [227]:
at = model_dict['At']

In [228]:
at_df = test_balance(at)

In [229]:
len(at_df)

18

In [230]:
j = 0
count_dict = defaultdict(int)
for i, row in at_df.iterrows():
    r_id = row['r_id']
    r = at.reactions[r_id]
    print(r, row['charge'])
    for met_id, coeff in r.stoichiometry.items():
        met = at.metabolites[met_id]
        count_dict[met_id] += 1
        print(f'  {met_id}: {coeff}, CHARGE={met.metadata.get("CHARGE", "NA")}')    
    j+=1
    # if j > 10:
    #     break

R_23CTI1: M_decoa_c --> M_dc2coa_c + M_h_c 1.0
  M_decoa_c: -1.0, CHARGE=-4
  M_dc2coa_c: 1.0, CHARGE=-4
  M_h_c: 1.0, CHARGE=1
R_ACOAD10f: M_fad_c + M_hptcoa_c + 2.0 M_h_c <-> M_fadh2_c + M_hp2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_hptcoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_hp2coa_c: 1.0, CHARGE=-4
R_ACOAD12f: M_fad_c + M_pdcacoa_c + 2.0 M_h_c <-> M_fadh2_c + M_pdca2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_pdcacoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_pdca2coa_c: 1.0, CHARGE=-4
R_ACOAD14f: M_fad_c + M_poctacoa_c + 2.0 M_h_c <-> M_fadh2_c + M_pocta2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_poctacoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_pocta2coa_c: 1.0, CHARGE=-4
R_ACOAD15f: M_fad_c + M_phpcoa_c + 2.0 M_h_c <-> M_fadh2_c + M_php2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_phpcoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_php2coa_c: 1.0, CHAR

In [231]:
# Get sorted count dict
sorted_counts = sorted(count_dict.items(), key=lambda x: x[1], reverse=True)

In [232]:
sorted_counts

[('M_h_c', 13),
 ('M_fad_c', 10),
 ('M_fadh2_c', 10),
 ('M_h2o_c', 4),
 ('M_atp_c', 2),
 ('M_adp_c', 2),
 ('M_pi_c', 2),
 ('M_3hnonacoa_c', 2),
 ('M_decoa_c', 1),
 ('M_dc2coa_c', 1),
 ('M_hptcoa_c', 1),
 ('M_hp2coa_c', 1),
 ('M_pdcacoa_c', 1),
 ('M_pdca2coa_c', 1),
 ('M_poctacoa_c', 1),
 ('M_pocta2coa_c', 1),
 ('M_phpcoa_c', 1),
 ('M_php2coa_c', 1),
 ('M_phxacoa_c', 1),
 ('M_phxa2coa_c', 1),
 ('M_pptcoa_c', 1),
 ('M_ppt2coa_c', 1),
 ('M_pbcoa_c', 1),
 ('M_pb2coa_c', 1),
 ('M_tdecoa_c', 1),
 ('M_tde2coa_c', 1),
 ('M_hdd4coa_c', 1),
 ('M_hdd4_2_coa_c', 1),
 ('M_dec4coa_c', 1),
 ('M_dec4_2_coa_c', 1),
 ('M_2dmmq8_c', 1),
 ('M_amet_c', 1),
 ('M_ahcys_c', 1),
 ('M_mql8_c', 1),
 ('M_fe3dhbzs3_p', 1),
 ('M_fe3dhbzs3_c', 1),
 ('M_nona2coa_c', 1),
 ('M_nad_c', 1),
 ('M_3ononacoa_c', 1),
 ('M_nadh_c', 1),
 ('M_2mecdp_c', 1),
 ('M_fdxrd_c', 1),
 ('M_fdxo_2_2_c', 1),
 ('M_h2mb4p_c', 1),
 ('M_dmlz_c', 1),
 ('M_4r5au_c', 1),
 ('M_ribflv_c', 1),
 ('M_salchs4fe_p', 1),
 ('M_salchs4fe_c', 1)]